In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb


train_model = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/processed/train_model.csv", parse_dates=["date"])
test_fe     = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/processed/test_fe.csv", parse_dates=["date"])

print(train_model.shape, test_fe.shape)

(2901096, 57) (28512, 56)


In [2]:
# fecha corte
cutoff = train_model["date"].max() - pd.Timedelta(days=28)

train_tr = train_model[train_model["date"] <= cutoff].copy()
train_va = train_model[train_model["date"] > cutoff].copy()

print(train_tr["date"].min(), train_tr["date"].max())
print(train_va["date"].min(), train_va["date"].max())

2013-02-26 00:00:00 2017-07-18 00:00:00
2017-07-19 00:00:00 2017-08-15 00:00:00


In [3]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)  # por seguridad
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

In [4]:
simple_cols = [
    "sales_lag_1","sales_lag_7","sales_lag_14","sales_lag_28","sales_lag_56",
    "promo_lag_1","promo_lag_7","promo_lag_28",
    "is_payday","holiday_weekday","holiday_weekend",
    "oil_above_mean","is_earthquake_window","dom_bucket_5"
]
simple_cols += [c for c in train_model.columns if c.startswith("dow_")]
simple_cols = [c for c in simple_cols if c in train_model.columns]

In [5]:
X_va = train_va[simple_cols].fillna(0)

max_per_col = X_va.max().sort_values(ascending=False).head(20)
max_per_col

sales_lag_56            24744.0
sales_lag_28            21180.0
sales_lag_1             18340.0
sales_lag_14            18340.0
sales_lag_7             18340.0
promo_lag_28              591.0
promo_lag_1               519.0
promo_lag_7               519.0
dom_bucket_5                6.0
dow_0                       1.0
dow_5                       1.0
dow_4                       1.0
dow_2                       1.0
dow_1                       1.0
holiday_weekend             1.0
holiday_weekday             1.0
is_payday                   1.0
dow_6                       1.0
is_earthquake_window        0.0
oil_above_mean              0.0
dtype: float64

In [6]:
X_va.describe().T.sort_values("max", ascending=False).head(20)[["min","mean","max"]]

,min,mean,max
sales_lag_56,0.0,493.217058,24744.0
sales_lag_28,0.0,477.061445,21180.0
sales_lag_1,0.0,469.062437,18340.0
sales_lag_14,0.0,475.761454,18340.0
sales_lag_7,0.0,477.100243,18340.0
promo_lag_28,0.0,8.144180,591.0
promo_lag_1,0.0,6.338203,519.0
promo_lag_7,0.0,6.648609,519.0
dom_bucket_5,0.0,2.571429,6.0
dow_0,0.0,0.142857,1.0


In [7]:
pred_va = 0.7*train_va["sales_lag_7"] + 0.3*train_va["sales_lag_14"]
mask = pred_va.notna()
print("Weighted lag RMSLE:", rmsle(train_va.loc[mask,"sales"], pred_va.loc[mask]))

Weighted lag RMSLE: 0.49855145243814497


In [8]:
pred_va2 = 0.5*train_va["sales_lag_1"] + 0.5*train_va["sales_lag_7"]
mask = pred_va2.notna()
print("Lag1+Lag7 RMSLE:", rmsle(train_va.loc[mask,"sales"], pred_va2.loc[mask]))

Lag1+Lag7 RMSLE: 0.4610427719606152


In [9]:
feature_cols = simple_cols  # simple

X_tr = train_tr[feature_cols].fillna(0)
y_tr = train_tr["y_log"]

X_va = train_va[feature_cols].fillna(0)
y_va = train_va["sales"].values

model = lgb.LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_tr, y_tr)

pred = np.expm1(model.predict(X_va))
pred = np.maximum(pred, 0)

print("LightGBM RMSLE:", rmsle(y_va, pred))
print("Baseline lag1+lag7 RMSLE:", rmsle(y_va, (0.5*train_va["sales_lag_1"] + 0.5*train_va["sales_lag_7"]).values))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036287 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1910
[LightGBM] [Info] Number of data points in the train set: 2851200, number of used features: 20
[LightGBM] [Info] Start training from score 2.945020
LightGBM RMSLE: 0.4119639060245271
Baseline lag1+lag7 RMSLE: 0.4610427719606152


In [13]:
ws = np.linspace(0, 1, 21)
best = (None, 1e9)

for w in ws:
    pred = w*train_va["sales_lag_1"] + (1-w)*train_va["sales_lag_7"]
    m = pred.notna()
    score = rmsle(train_va.loc[m,"sales"], pred.loc[m])
    if score < best[1]:
        best = (w, score)
best

(np.float64(0.45), np.float64(0.46058508573925044))

In [14]:
w = 0.45
pred_opt = w*train_va["sales_lag_1"] + (1-w)*train_va["sales_lag_7"]
m = pred_opt.notna()
print("Optimal weighted baseline RMSLE:", rmsle(train_va.loc[m,"sales"], pred_opt.loc[m]))

Optimal weighted baseline RMSLE: 0.46058508573925044


In [11]:
# train final
X_full = train_model[feature_cols].fillna(0)
y_full = train_model["y_log"]

model.fit(X_full, y_full)

# predict test
X_test = test_fe[feature_cols].fillna(0)
test_pred = np.expm1(model.predict(X_test))
test_pred = np.maximum(test_pred, 0)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030093 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1914
[LightGBM] [Info] Number of data points in the train set: 2901096, number of used features: 20
[LightGBM] [Info] Start training from score 2.956567


In [16]:
sample = pd.read_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/raw/store-sales-time-series-forecasting/sample_submission.csv")

sub = sample.copy()
sub["sales"] = test_pred

# sanity checks
print(sub.shape)
print(sub.head())

sub.to_csv("/Users/JulioJerez/Desktop/Docs. Personales/Repositorio/PROYECTOS/store-sales-forecasting/data/processed/submission_lgbm.csv", index=False)

(28512, 2)
        id     sales
0  3000888  4.068202
1  3000889  2.196262
2  3000890  2.115397
3  3000891  4.112711
4  3000892  1.319283


In [17]:
print("Any NaN?", sub["sales"].isna().any())
print("Any negative?", (sub["sales"] < 0).any())
print("min/median/max:", sub["sales"].min(), sub["sales"].median(), sub["sales"].max())

Any NaN? False
Any negative? False
min/median/max: 0.0 1.472149025182326 11728.440730659551


In [18]:
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
imp.head(15)

sales_lag_1        7271
sales_lag_7        6988
sales_lag_56       6495
sales_lag_28       5570
sales_lag_14       5515
dom_bucket_5       3676
promo_lag_1        2627
promo_lag_7        2328
promo_lag_28       2158
holiday_weekday    1255
dow_6              1091
dow_0               998
oil_above_mean      954
dow_5               709
dow_4               663
dtype: int32